# Desafio 3 — Normalização de JSONs Aninhados: Dados de Países (REST Countries API)

**Conceitos praticados:** `explode()`, `explode(map_entries())`, `MapType`, `ArrayType`,
`StructType` aninhado, tabelas dimensão no modelo Star Schema, `MERGE INTO` idempotente.

**Seções do exame:** S2 (Data Ingestion) e S3 (Data Transformation and Modeling)

**API:** https://restcountries.com/v3.1/all — 100% pública, sem chave, ~250 países.

### Analogia
O JSON da REST Countries é como um armário de arquivos onde cada gaveta (país) contém
pastas dentro de pastas: uma pasta "Idiomas" com subpastas por código ISO, uma pasta
"Moedas" com símbolo e nome aninhados. Para trabalhar em SQL, você precisa "desdobrar"
esse armário em planilhas planas. Isso é o `explode()`.

### Tipos especiais no JSON que vamos tratar:
- `languages` e `currencies` → **MapType** (chave ISO → valor) → `explode(map_entries(...))`
- `capital`, `timezones`, `borders` → **ArrayType** → `capital[0]`, `explode()`
- `latlng` → array de 2 elementos → `latlng[0]` (lat), `latlng[1]` (lon)
- `capital` pode ser **array vazio** → tratar com `CASE WHEN size(capital) > 0`

> ⚠️ Se o outbound internet estiver bloqueado, use o **Plano B** documentado na Célula 4.

In [ ]:
%sql
-- Passo 0: criar o schema countries (execute uma única vez)
CREATE SCHEMA IF NOT EXISTS workshop_weather.countries
  COMMENT 'Dados geográficos e demográficos — REST Countries API';

In [ ]:
# ── Imports e configuração ──────────────────────────────────────────────────
import requests
import json
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType,
    DateType, TimestampType, ArrayType, MapType
)
from datetime import datetime

CATALOG = "workshop_weather"
SCHEMA  = "countries"
TABLE_BRONZE      = f"{CATALOG}.{SCHEMA}.raw_restcountries"
TABLE_DIM_COUNTRY = f"{CATALOG}.{SCHEMA}.dim_country"
TABLE_DIM_LANG    = f"{CATALOG}.{SCHEMA}.dim_country_language"
TABLE_DIM_CURR    = f"{CATALOG}.{SCHEMA}.dim_country_currency"
TABLE_GOLD        = f"{CATALOG}.{SCHEMA}.gold_south_america_profile"

print(f"Bronze:             {TABLE_BRONZE}")
print(f"Silver dim_country: {TABLE_DIM_COUNTRY}")
print(f"Silver dim_lang:    {TABLE_DIM_LANG}")
print(f"Silver dim_curr:    {TABLE_DIM_CURR}")
print(f"Gold:               {TABLE_GOLD}")

### ⚠️ Plano B — API inacessível (outbound bloqueado)

Execute o script abaixo **localmente** na sua máquina e faça upload do arquivo gerado.

```python
# Execute LOCALMENTE (não no Databricks):
import requests, json

# Opção A — todos os países (~250 países, ~2MB)
r = requests.get("https://restcountries.com/v3.1/all")
with open("countries_all.json", "w", encoding="utf-8") as f:
    json.dump(r.json(), f, ensure_ascii=False)
print(f"Salvo: countries_all.json ({len(r.json())} países)")

# Opção B — apenas Américas (menor, bom para teste inicial)
# r = requests.get("https://restcountries.com/v3.1/region/americas")
# with open("countries_americas.json", "w", encoding="utf-8") as f:
#     json.dump(r.json(), f, ensure_ascii=False)
```

Faça upload para: **Data > Volumes > workshop_weather > bronze > landing > countries**

Ajuste `USE_PLAN_B = True` e `PLAN_B_PATH` na próxima célula.

In [ ]:
# ── Configuração Plano B ─────────────────────────────────────────────────────
USE_PLAN_B  = False
PLAN_B_PATH = "/Volumes/workshop_weather/bronze/landing/countries/countries_all.json"
# Para Américas: ".../countries_americas.json"
# ────────────────────────────────────────────────────────────────────────────

ingestion_date = datetime.utcnow().date().isoformat()

if not USE_PLAN_B:
    url = "https://restcountries.com/v3.1/all"
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        countries_list = response.json()
        print(f"[OK] {len(countries_list)} países carregados da API.")
    except Exception as e:
        raise RuntimeError(f"API falhou: {e}\nAtive USE_PLAN_B = True.")
else:
    print(f"[PLANO B] Lendo: {PLAN_B_PATH}")
    raw_text = spark.read.format("text").option("wholetext", "true").load(PLAN_B_PATH).collect()[0][0]
    countries_list = json.loads(raw_text)
    print(f"[PLANO B] {len(countries_list)} países carregados do arquivo local.")

In [ ]:
# ── Grava Bronze: payload completo como string + metadados ───────────────────
# Decisão de design: 1 linha por PAÍS na Bronze
# (alternativa: 1 linha com todo o array — comentada abaixo)
# Escolhemos 1 linha por país para facilitar o reprocessamento individual.

rows_bronze = [
    (
        country.get("cca3", "UNKNOWN"),   # country_code_3 — identificador de rastreabilidade
        json.dumps(country),               # raw_payload — objeto JSON completo como string
        ingestion_date,                    # _ingestion_date
    )
    for country in countries_list
    if country.get("cca3")                 # exclui entradas sem código ISO3
]

bronze_schema = StructType([
    StructField("country_code_3",   StringType(), False),
    StructField("raw_payload",      StringType(), False),
    StructField("_ingestion_date",  StringType(), False),
])

df_bronze = spark.createDataFrame(rows_bronze, schema=bronze_schema)
df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_BRONZE)

print(f"[OK] Bronze gravada: {df_bronze.count()} países")
spark.sql(f"SELECT country_code_3, LEFT(raw_payload, 80) as preview FROM {TABLE_BRONZE} LIMIT 3").show(truncate=False)

## Passo 2 — Silver: 3 tabelas dimensão normalizadas

Vamos extrair 3 tabelas a partir do mesmo payload Bronze:

| Tabela | Grão | Técnica PySpark |
|---|---|---|
| `dim_country` | 1 linha por país | `get_json_object()`, `from_json()` |
| `dim_country_language` | 1 linha por (país, idioma) | `explode(map_entries(languages))` |
| `dim_country_currency` | 1 linha por (país, moeda) | `explode(map_entries(currencies))` |

### Por que tabelas separadas?
Se `languages` fosse uma coluna array em `dim_country`, você não conseguiria fazer:
`WHERE language_name = 'Portuguese'` de forma eficiente.
Normalizar em tabela separada segue a **3ª Forma Normal** e permite JOINs limpos.

In [ ]:
# ── Silver: dim_country — schema plano, 1 linha por país ────────────────────
from pyspark.sql.functions import (
    get_json_object, col, lit, to_date, current_date,
    when, size, element_at
)

df_raw = spark.read.format("delta").table(TABLE_BRONZE)

# Schema explícito para o JSON de cada país
country_json_schema = StructType([
    StructField("name", StructType([
        StructField("common",   StringType(), True),
        StructField("official", StringType(), True),
    ]), True),
    StructField("cca2",       StringType(), True),
    StructField("cca3",       StringType(), True),
    StructField("region",     StringType(), True),
    StructField("subregion",  StringType(), True),
    StructField("population", LongType(),   True),
    StructField("area",       DoubleType(), True),
    StructField("latlng",     ArrayType(DoubleType()), True),
    StructField("capital",    ArrayType(StringType()), True),
])

# Parse do JSON com schema explícito
df_parsed = df_raw.withColumn("c", F.from_json(col("raw_payload"), country_json_schema))

# Extração dos campos com tratamento de casos especiais:
# - capital pode ser array vazio → retorna NULL
# - latlng pode ser None em alguns territórios → retorna NULL com coalesce
df_dim_country = df_parsed.select(
    col("c.cca2").alias("country_code_2"),
    col("c.cca3").alias("country_code_3"),
    col("c.name.common").alias("country_name_common"),
    col("c.name.official").alias("country_name_official"),
    col("c.region").alias("region"),
    col("c.subregion").alias("subregion"),
    col("c.population").alias("population"),
    col("c.area").alias("area_km2"),
    # latlng[0] = latitude, latlng[1] = longitude
    when(col("c.latlng").isNotNull() & (size(col("c.latlng")) >= 2),
         col("c.latlng")[0]).otherwise(F.lit(None).cast(DoubleType())).alias("latitude"),
    when(col("c.latlng").isNotNull() & (size(col("c.latlng")) >= 2),
         col("c.latlng")[1]).otherwise(F.lit(None).cast(DoubleType())).alias("longitude"),
    # capital[0] — array pode ser vazio em alguns territórios
    when(col("c.capital").isNotNull() & (size(col("c.capital")) > 0),
         col("c.capital")[0]).otherwise(F.lit(None)).alias("capital"),
    F.to_date(F.lit(ingestion_date)).alias("_ingestion_date"),
).filter(col("country_code_3").isNotNull())

print("[INFO] Schema da dim_country:")
df_dim_country.printSchema()
print(f"\n[INFO] {df_dim_country.count()} países extraídos")

In [ ]:
# ── Cria a tabela e executa MERGE INTO dim_country ───────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_DIM_COUNTRY} (
    country_code_2        STRING,
    country_code_3        STRING NOT NULL,
    country_name_common   STRING,
    country_name_official STRING,
    region                STRING,
    subregion             STRING,
    population            LONG,
    area_km2              DOUBLE,
    latitude              DOUBLE,
    longitude             DOUBLE,
    capital               STRING,
    _ingestion_date       DATE
)
USING DELTA
COMMENT 'Dimensão de países — REST Countries API'
""")

df_dim_country.createOrReplaceTempView("staging_dim_country")

spark.sql(f"""
MERGE INTO {TABLE_DIM_COUNTRY} AS target
USING staging_dim_country AS source
ON target.country_code_3 = source.country_code_3
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

print(f"[OK] MERGE INTO dim_country executado.")
spark.sql(f"SELECT COUNT(*) as total FROM {TABLE_DIM_COUNTRY}").show()
spark.sql(f"SELECT * FROM {TABLE_DIM_COUNTRY} WHERE country_code_3 = 'BRA'").show(truncate=False)

## Silver: dim_country_language — `explode(map_entries(languages))`

O campo `languages` no JSON é um **MapType**: `{"por": "Portuguese", "spa": "Spanish"}`.

Para normalizar em linhas, usamos `explode(map_entries(...))`:
- `map_entries()` converte o mapa em um array de structs `{key, value}`
- `explode()` quebra o array em uma linha por elemento

Resultado: 1 linha por (país, idioma). Um país com 3 idiomas gera 3 linhas.

In [ ]:
# ── Silver: dim_country_language ─────────────────────────────────────────────
# languages no JSON: {"por": "Portuguese"} → MapType(StringType, StringType)
lang_schema = StructType([
    StructField("cca3",      StringType(), True),
    StructField("languages", MapType(StringType(), StringType()), True),
])

df_lang_raw = df_raw.withColumn(
    "parsed",
    F.from_json(col("raw_payload"), lang_schema)
    ).select(
    col("parsed.cca3").alias("country_code_3"),
    col("parsed.languages").alias("languages_map")
).filter(
    col("country_code_3").isNotNull() & col("languages_map").isNotNull()
)

# explode(map_entries()) — transforma o MapType em linhas
df_dim_lang = df_lang_raw.select(
    col("country_code_3"),
    F.explode(F.map_entries(col("languages_map"))).alias("lang_entry")
).select(
    col("country_code_3"),
    col("lang_entry.key").alias("language_code"),
    col("lang_entry.value").alias("language_name")
)

print("[INFO] Schema dim_country_language:")
df_dim_lang.printSchema()
print(f"\n[INFO] Total de registros (países × idiomas): {df_dim_lang.count()}")
print("\nTop 5 países com mais idiomas:")
df_dim_lang.groupBy("country_code_3").count().orderBy(F.desc("count")).show(5)
print("\nIdiomas do Brasil:")
df_dim_lang.filter(col("country_code_3") == "BRA").show()

In [ ]:
# ── MERGE INTO dim_country_language ──────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_DIM_LANG} (
    country_code_3  STRING NOT NULL,
    language_code   STRING NOT NULL,
    language_name   STRING
)
USING DELTA
COMMENT 'Idiomas por país — normalizado com explode(map_entries)'
""")

df_dim_lang.createOrReplaceTempView("staging_dim_lang")

spark.sql(f"""
MERGE INTO {TABLE_DIM_LANG} AS target
USING staging_dim_lang AS source
ON  target.country_code_3 = source.country_code_3
AND target.language_code  = source.language_code
WHEN MATCHED THEN UPDATE SET language_name = source.language_name
WHEN NOT MATCHED THEN INSERT *
""")

print(f"[OK] MERGE INTO dim_country_language executado.")
spark.sql(f"SELECT COUNT(*) as total FROM {TABLE_DIM_LANG}").show()

## Silver: dim_country_currency — `explode(map_entries(currencies))`

O campo `currencies` tem um nível a mais de aninhamento:
`{"BRL": {"name": "Brazilian real", "symbol": "R$"}}`

Precisamos:
1. `explode(map_entries(currencies))` → 1 linha por moeda por país
2. Acessar `entry.value.name` e `entry.value.symbol` do struct aninhado

In [ ]:
# ── Silver: dim_country_currency ─────────────────────────────────────────────
# currencies no JSON: {"BRL": {"name": "Brazilian real", "symbol": "R$"}} 
# → MapType(StringType, StructType({name, symbol}))

currency_value_schema = StructType([
    StructField("name",   StringType(), True),
    StructField("symbol", StringType(), True),
])

curr_schema = StructType([
    StructField("cca3",       StringType(), True),
    StructField("currencies", MapType(StringType(), currency_value_schema), True),
])

df_curr_raw = df_raw.withColumn(
    "parsed",
    F.from_json(col("raw_payload"), curr_schema)
).select(
    col("parsed.cca3").alias("country_code_3"),
    col("parsed.currencies").alias("currencies_map")
).filter(
    col("country_code_3").isNotNull() & col("currencies_map").isNotNull()
)

# explode(map_entries()) no mapa de moedas
df_dim_curr = df_curr_raw.select(
    col("country_code_3"),
    F.explode(F.map_entries(col("currencies_map"))).alias("curr_entry")
).select(
    col("country_code_3"),
    col("curr_entry.key").alias("currency_code"),
    col("curr_entry.value.name").alias("currency_name"),
    col("curr_entry.value.symbol").alias("currency_symbol"),
)

print("[INFO] Schema dim_country_currency:")
df_dim_curr.printSchema()
print(f"\n[INFO] Total de registros (países × moedas): {df_dim_curr.count()}")
print("\nMoedas do Brasil:")
df_dim_curr.filter(col("country_code_3") == "BRA").show(truncate=False)

In [ ]:
# ── MERGE INTO dim_country_currency ──────────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_DIM_CURR} (
    country_code_3  STRING NOT NULL,
    currency_code   STRING NOT NULL,
    currency_name   STRING,
    currency_symbol STRING
)
USING DELTA
COMMENT 'Moedas por país — normalizado com explode(map_entries)'
""")

df_dim_curr.createOrReplaceTempView("staging_dim_curr")

spark.sql(f"""
MERGE INTO {TABLE_DIM_CURR} AS target
USING staging_dim_curr AS source
ON  target.country_code_3 = source.country_code_3
AND target.currency_code  = source.currency_code
WHEN MATCHED THEN UPDATE SET
    currency_name   = source.currency_name,
    currency_symbol = source.currency_symbol
WHEN NOT MATCHED THEN INSERT *
""")

print(f"[OK] MERGE INTO dim_country_currency executado.")
spark.sql(f"SELECT COUNT(*) as total FROM {TABLE_DIM_CURR}").show()

## Passo 3 — Gold: Perfil da América do Sul

JOIN das 3 tabelas Silver para montar o perfil completo dos países da América do Sul,
com idiomas e moedas consolidados em listas.

In [ ]:
# ── Gold: perfil da América do Sul ───────────────────────────────────────────
df_gold = spark.sql(f"""
WITH languages_agg AS (
    SELECT
        country_code_3,
        collect_list(language_name) AS languages_list
    FROM {TABLE_DIM_LANG}
    GROUP BY country_code_3
),
currencies_agg AS (
    SELECT
        country_code_3,
        collect_list(CONCAT(currency_code, ' (', COALESCE(currency_symbol, '-'), ')')) AS currencies_list
    FROM {TABLE_DIM_CURR}
    GROUP BY country_code_3
)
SELECT
    c.country_name_common,
    c.country_code_3,
    c.capital,
    c.population,
    ROUND(c.area_km2, 0)     AS area_km2,
    c.latitude,
    c.longitude,
    l.languages_list,
    cu.currencies_list
FROM {TABLE_DIM_COUNTRY} c
LEFT JOIN languages_agg  l  ON c.country_code_3 = l.country_code_3
LEFT JOIN currencies_agg cu ON c.country_code_3 = cu.country_code_3
WHERE c.subregion = 'South America'
ORDER BY c.population DESC
""")

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_GOLD)

print(f"[OK] Gold gravada: {df_gold.count()} países da América do Sul")
display(spark.table(TABLE_GOLD))

In [ ]:
%sql
-- Otimiza a tabela de países para queries por região
OPTIMIZE workshop_weather.countries.dim_country
ZORDER BY (region, country_code_3);

In [ ]:
print("=" * 60)
print("VALIDAÇÃO FINAL — Desafio 3")
print("=" * 60)

bronze_n = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_BRONZE}").collect()[0][0]
dim_c_n  = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_DIM_COUNTRY}").collect()[0][0]
dim_l_n  = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_DIM_LANG}").collect()[0][0]
dim_cu_n = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_DIM_CURR}").collect()[0][0]
gold_n   = spark.sql(f"SELECT COUNT(*) as n FROM {TABLE_GOLD}").collect()[0][0]

print(f"Bronze (raw_restcountries):       {bronze_n:,} linhas")
print(f"Silver (dim_country):             {dim_c_n:,} países")
print(f"Silver (dim_country_language):    {dim_l_n:,} pares país-idioma")
print(f"Silver (dim_country_currency):    {dim_cu_n:,} pares país-moeda")
print(f"Gold   (gold_south_america):      {gold_n} países (esperado: ~12)")

print("""
PERGUNTAS DE FIXAÇÃO (responda mentalmente):

1. Por que `dim_country_language` é uma tabela separada e não um
   array dentro de `dim_country`? Qual problema de modelagem isso resolve?

2. Qual é a diferença entre usar `explode()` em um ArrayType
   versus `explode(map_entries())` em um MapType?

3. Se você tivesse usado `get_json_object(raw_payload, "$.languages")`
   em vez de `from_json()` com schema explícito, o que retornaria?
   Em qual situação `get_json_object` seria mais prático?

4. O que acontece com `capital[0]` quando o array de capital está vazio?
   Como tratamos isso no notebook?

5. Como o exame diferencia `explode()` de `posexplode()`?
   Quando usaria `posexplode()`?
""")